In [5]:
import pandas as pd

interactions = pd.read_parquet("data\\interactions.parquet")

interactions.head(10)

,user_idx,movie_idx,rating
0,0,1104,5.0
1,0,639,3.0
2,0,853,3.0
3,0,3177,4.0
4,0,2162,5.0
5,0,1107,3.0
6,0,1195,5.0
7,0,2599,5.0
8,0,580,4.0
9,0,858,4.0


In [6]:
def recommend_popular(
        user_idx: int,
        interactions: pd.DataFrame,
        k: int = 10
):
    popularity = (
        interactions.groupby("movie_idx").rating
        .agg(
            rating_count="count",
            avg_rating="mean"
        )
        .reset_index()
        .sort_values(by=["rating_count", "avg_rating"], ascending=False)
    )

    watched_movies = set(
        interactions.loc[
            interactions.user_idx == user_idx, "movie_idx"
        ]
    )

    

    return (
        popularity[
            ~popularity.movie_idx.isin(watched_movies)
        ]
        .head(k)
    )

print(recommend_popular(user_idx=3, interactions=interactions, k=10))


      movie_idx  rating_count  avg_rating
2651       2651          3428    4.317386
575         575          2649    4.058513
2374       2374          2590    4.315830
1178       1178          2583    3.990321
579         579          2578    4.351823
1449       1449          2538    3.739953
593         593          2513    4.254676
2557       2557          2459    4.406263
106         106          2443    4.234957
2203       2203          2369    4.127480


In [15]:
def bayesian_popularity(
    interactions: pd.DataFrame,
    m: int = 100
):
    C = interactions.rating.mean()

    popularity = (
        interactions.groupby("movie_idx").rating
        .agg(
            rating_count="count",
            avg_rating="mean"
        )
        .reset_index()
    )

    popularity["score"] = (
        popularity.rating_count 
        / (popularity.rating_count + m) 
        * popularity.avg_rating 
        + m 
        / (popularity.rating_count + m)
        * C
    )

    return popularity.sort_values(by="score", ascending=False)

def recommend_bayesian_popularity(
    user_idx: int,
    popularity: pd.DataFrame,
    interactions: pd.DataFrame,
    k: int = 10
):
    
    watched_movies = set(
        interactions.loc[
            interactions.user_idx == user_idx, "movie_idx"
        ]
    )

    return (
        popularity[
            ~popularity.movie_idx.isin(watched_movies)
        ]
        .reset_index()
        .head(k)
        .sort_values(by="score", ascending=False)
        .movie_idx
    )


model = bayesian_popularity(interactions=interactions, m=100)

print(recommend_bayesian_popularity(1, model, interactions, k=10))

0     802
1     513
2      49
3    1839
4     253
5    1066
6     843
7     708
8     713
9    2557
Name: movie_idx, dtype: int64


In [18]:
train_parts = []
test_parts = []

for _, group in interactions.groupby("user_idx"):
    train_part = group.sample(frac=0.8, random_state=42)
    test_part = group.drop(train_part.index)

    train_parts.append(train_part)
    test_parts.append(test_part)

train = pd.concat(train_parts).reset_index(drop=True)
test = pd.concat(test_parts).reset_index(drop=True)

In [ ]:
import sys
from pathlib import Path
import numpy as np

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

from src.metrics.metrics import (
    recall_at_k,
    precision_at_k,
    ndcg_at_k,
)

recalls = []
precisions = []
ndcgs = []

k = 10

for user_idx in interactions.user_idx.unique():

    recommended = recommend_bayesian_popularity(user_idx, model, train).tolist()
    test_user = test[test.user_idx == user_idx]

    relevant = test_user.loc[
        test_user.rating >= 4,
        "movie_idx"
    ].tolist()

    if len(relevant) == 0:
        continue

    recalls.append(recall_at_k(relevant, recommended, k))
    precisions.append(precision_at_k(relevant, recommended, k))
    ndcgs.append(ndcg_at_k(relevant, recommended, k))

recalls = np.array(recalls)
precisions = np.array(precisions)
ndcgs = np.array(ndcgs)

print(
    f"recall at k: {recalls.mean()}\n"
    f"precision at k: {precisions.mean()}\n"
    f"ndcg at k: {ndcgs.mean()}"
)

recall at k: 0.04803111990676727
precision at k: 0.0821007146418481
ndcg at k: 0.09873952472835491


: 